---
title: Executor inputs and outputs
description: Understand the inputs and outputs to the Executor primitive.
---


# Executor inputs and outputs

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

The Executor primitive is part of the [directed execution model](/docs/guides/directed-execution-model), which helps provide more flexibility when customizing an error mitigation workflow.

The inputs and output of the Executor primitive are very different from those of the [Sampler](/docs/guides/sampler-input-output) and [Estimator](/docs/guides/estimator-input-output) primitives. For example, instead of taking a list of PUBs as the input, Executor takes a `QuantumProgram`, which contains a list of `QuantumProgramItem` objects. `QuantumProgramItem` objects can be of two types:
- `CircuitItem`, which typically stores a circuit and its parameters.
- `SamplexItem`, which typically stores a circuit, its parameters, and a `Samplex` object. The samplex is capable of generating randomized sets of parameters at runtime, for example to perform twirling or inject noise.

These container classes give you more flexibility than a PUB, which is a simple tuple data structure. For example, they allow you to specify where and how a set of circuit instructions should be randomized before execution.

Executor's output is a `QuantumProgramResult`, which is an iterable and contains one element for each input `QuantumProgramItem`. Each element is a dictionary whose keys correspond the classical registers' names in the input circuits (among others), so you no longer need to memorize these names like you did with Sampler output.

This primitive takes a `QuantumProgram` as input, and then outputs a Qiskit Runtime job, which is then run on an IBM&reg; quantum computer.

<span id="programs"></span>
## Inputs: Quantum programs

The input to an Executor primitive is a  [`QuantumProgram`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgram.html#QuantumProgram), which is an iterable of a
[`QuantumProgramItem`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramItem.html). Each of these items represents a
different task for Executor to perform. Typically, each item owns one of the following options:

* A `QuantumCircuit` with static, non-parametrized gates
* A parametrized `QuantumCircuit` with an array of parameter values
* A parametrized `QuantumCircuit` with a `Samplex`. A samplex is a core type of the [Samplomatic library](https://github.com/Qiskit/samplomatic) and encodes all the information about the randomization process itself.

Each of these items, including instructions to add them to a `QuantumProgram`, is explained in more detail below.

### Example: Create a `QuantumProgram` with two different tasks

First initialize your quantum program, then append program items to it by using either `append_circuit_item` or `append_samplex_item` (if a samplex is present), as shown in the following examples.

The following cell initializes a `QuantumProgram` and specifies that it should run 1024 shots for every configuration of each item in the program. Then it appends a version of the target circuit with parameters, transpiled according to the backend's instruction set architecture (ISA).

<Admonition type="note">
Unlike with Sampler, a `QuantumProgram` takes only a single shot value. If you want a different shot value, you need a separate `QuantumProgram`, which would be a separate job.
</Admonition>

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_ibm_runtime import Executor, QiskitRuntimeService
from qiskit.circuit import Parameter, QuantumCircuit
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Initialize an empty program
program = QuantumProgram(shots=1024)

# Initialize and transpile a 3-qubit quantum circuit with 3 parameters.
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)
circuit.rz(Parameter("lam"), 2)

In [2]:
# `measure_all` adds a 3-bit classical register named "meas"
circuit.measure_all()

# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an
# equivalent Instruction Set Architecture (ISA) circuit.
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Transpile the circuit
isa_circuit = preset_pass_manager.run(circuit)

# Append the transpiled circuit and an array
# containing 10 sets of parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(10, 3),  # 10 sets of parameter values)
)

Circuit items are executed without any sort of randomization. On the contrary, samplex items let you specify how to randomize their content. The next cell generates a samplex item for the GHZ circuit that adds twirling to all of the circuit's gates and measurements.

See the Samplomatic [API](https://github.com/Qiskit/samplomatic/) documentation for full details about `Samplex` and its arguments.

In [3]:
# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Use the boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template and the samplex. The template circuit is a
# parametric circuit, and the samplex is an object that can generate
# randomized sets of parameters at runtime to perform twirling
template, samplex = build(boxed_circuit)

# Append the template and samplex as a samplex item
program.append_samplex_item(
    template,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 3),
    },
    shape=(28, 10),  # 28 randomizations
)

In [4]:
# initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Outputs

Executor's output is a [`QuantumProgramResult`,](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramResult.html#QuantumProgramResult) which is an iterable. It contains one
item per circuit task, and the items are in the same order as those in the program. Each of
these items is a dictionary where the keys are strings and the values are of type `np.ndarray`.

The result for the previous example contains these items:

### Result item 0

The first item contains the results of running the second task in the program,
the circuit with parametrized gates.

It contains a single key, `'meas'`, mapped to an `np.ndarray` of shape `(shots, parameter sets, register bits)`, which is `(1024, 10, 3)` for the above example.

The following code illustrates how to access this information:

In [5]:
# Access the results of the classical register of task #0
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

Result shape: (28, 10, 1024, 3)


### Result item 1

The second item contains the results of running the third task in the program. This item
contains multiple keys. In addition to the `'meas'` key (mapped to the array of results for
that classical register), it contains `'measurement_flips.meas'`, the bit-flip corrections to undo
the measurement twirling for the `'meas'` register, as follows:

In [6]:
# Access the results of the classical register of task #1
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

# Access the bit-flip corrections
flips_1 = result[1]["measurement_flips.meas"]
print(f"Bit-flip corrections shape: {flips_1.shape}")

# Undo the bit flips via classical XOR
unflipped_result_1 = result_1 ^ flips_1

Result shape: (28, 10, 1024, 3)
Bit-flip corrections shape: (28, 10, 1, 3)


## Next steps

<Admonition type="tip" title="Recommendations">
- Explore [examples](/docs/guides/executor-examples) that use Executor.
- Learn about the [directed execution model](/docs/guides/directed-execution-model).
- Understand [Executor broadcasting](/docs/guides/executor-broadcasting).
</Admonition>